# 08 · Reuse-category assessment (E10)

**Goal:** turn SOH estimates into reuse categories and measure how reliably the categories from predicted SOH agree with those from measured SOH (RQ4; `docs/REUSE_CATEGORIES.md` §4, decision D10).

**Categories (D10)**, on fade since the first observed RPT (`F = 100 − SOH`), fixed before any real-data run:

| Class | Rule on SOH |
|---|---|
| C0 stable | SOH > 95 |
| C1 moderate fade | 90 < SOH ≤ 95 |
| C2 substantial fade | 80 < SOH ≤ 90 |
| C3 severe fade | SOH ≤ 80 |

They describe capacity change since the first second-life measurement, **not** suitability for an application.

**Protocol**
* Classes are obtained by applying the rule to predicted SOH from the regression models. No separate classifier is trained.
* Headline unit: one sample per (battery, RPT) in the test period, using mean predicted SOH over the interval (removes the weighting by repeated labels, D4). Cycle-level results are also reported.
* Metrics: accuracy; macro-F1 with the number of classes present; linear-weighted kappa; share of errors of more than one class; confusion matrix; **boundary share** (fraction of samples whose measured SOH lies within ±MAE of a cut-point, i.e. classes the model cannot resolve at its error level).
* Baselines: majority class of each battery's training period, and persistence (class at the last training RPT).
* Models: the E1 within-group models from notebook 05 (Ridge, MLP, TFT). Category results after each continual-learning step are in notebooks 06–07 (`reuse_*` columns).
* Pre-registered sensitivity analyses: cut-points ±2.5 pp (S-a1), 3-class rule (S-a2), fade rate per 100 days (S-c), end-of-day date convention (S-D3). None is used for selection. S-b (absolute SOH) is run only if the data provenance is confirmed.

Set `EXPERIMENT_PROFILE=smoke` for a fast code check; smoke results are not for reporting.

## 1. Setup

In [ ]:
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd

# Resolve the repository root whether the kernel starts in the repo root or in notebooks/.
REPO_ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "src" / "data_pipeline.py").is_file())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src import data_pipeline as dp

DATA_ROOT = dp.resolve_data_root(os.environ.get("DATA_ROOT", "data"), REPO_ROOT)
RESULTS_DIR = REPO_ROOT / "results"

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 200)
print("DATA_ROOT env :", os.environ.get("DATA_ROOT", "<unset, default 'data'>"))
print("DATA_ROOT dir exists:", DATA_ROOT.is_dir())

import matplotlib.pyplot as plt

# Fixed colour per battery group (never re-assigned by rank or filter).
GROUP_COLORS = dict(zip(dp.BATTERY_GROUPS, ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300"]))
plt.rcParams.update({
    "figure.dpi": 110,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.color": "#e4e3dd",
    "grid.linewidth": 0.6,
    "axes.edgecolor": "#8a8981",
    "axes.titlesize": 11,
    "axes.labelsize": 10,
    "lines.linewidth": 2,
})
FIG_DIR = RESULTS_DIR / "figures"


def savefig(fig, name):
    """Save a figure for the paper/report (results/figures is regenerated, not committed)."""
    FIG_DIR.mkdir(parents=True, exist_ok=True)
    fig.savefig(FIG_DIR / f"{name}.png", dpi=150, bbox_inches="tight")

from src import evaluation as ev
from src import experiments as X
from src import protocols as P
from src import reuse as RU

PROFILE = os.environ.get("EXPERIMENT_PROFILE", "paper")
CONFIG = {
    "profile": PROFILE,
    "models": ["ridge", "mlp", "tft"],
    "seed": 0,
    "train_frac": 0.70,
    "val_frac": 0.15,
    "neural_kwargs": {"epochs": 200, "patience": 20} if PROFILE == "paper" else {"epochs": 8, "patience": 3},
    "cutpoint_variants": {k: list(v) for k, v in RU.CUTPOINT_VARIANTS.items()},
}
CLASS_SHORT = [c.split()[0] for c in ev.REUSE_CLASS_NAMES]
CLASS_RAMP = ["#cde2fb", "#6da7ec", "#2a78d6", "#104281"]  # ordinal classes: one hue, light -> dark
print(CONFIG)

## 2. Load data

In [ ]:
dp.check_data_files(DATA_ROOT, required=["ml_training_dataset.csv", "final_battery_health_dataset.csv"])
data = dp.read_dataset(DATA_ROOT / "ml_training_dataset.csv")
health = dp.read_dataset(DATA_ROOT / "final_battery_health_dataset.csv")
data[dp.GROUP_COL] = data[dp.GROUP_COL].astype(str)
groups = [g for g in dp.BATTERY_GROUPS if g in set(data[dp.GROUP_COL])]
print("rows:", len(data), "| groups:", groups)

## 3. Class distribution of measured SOH

How many RPT intervals fall in each class, over the whole labelled period and within the E1 test periods. Test periods containing a single measured class make macro-F1 uninterpretable for that group; they are flagged in the results rather than merged into other classes.

In [ ]:
train, val, tests = X.pooled_split(data, groups, CONFIG["train_frac"], CONFIG["val_frac"])
test_all = pd.concat(tests.values(), ignore_index=True)
dist_all = RU.class_distribution(data).reindex(groups)
dist_test = RU.class_distribution(test_all).reindex(groups).fillna(0).astype(int)
display(pd.concat({"all labelled RPT intervals": dist_all, "E1 test-period intervals": dist_test}, axis=1))

shares = dist_test.div(dist_test.sum(axis=1), axis=0)
fig, ax = plt.subplots(figsize=(8, 3.4))
left = np.zeros(len(groups))
for k, cls in enumerate(dist_test.columns):
    ax.barh(groups, shares[cls], left=left, color=CLASS_RAMP[k], height=0.6, label=cls, edgecolor="white", linewidth=2)
    left += shares[cls].to_numpy()
ax.invert_yaxis()
ax.set_xlim(0, 1)
ax.set_xlabel("Share of test-period RPT intervals")
ax.set_title("Measured reuse class in each group's test period", loc="left")
ax.grid(axis="y", visible=False)
ax.legend(frameon=False, ncol=4, fontsize=8, loc="upper center", bbox_to_anchor=(0.5, -0.22))
plt.tight_layout()
plt.show()

## 4. S-c: fade rate per battery (descriptive)

Least-squares slope of SOH against days since the first RPT, in percentage points per 100 days. Descriptive only; it contextualises why groups reach different classes within the observation period.

In [ ]:
h = health.assign(**{dp.GROUP_COL: health[dp.GROUP_COL].astype(str)}).sort_values([dp.GROUP_COL, "Test_Date"])
h["days"] = (h["Test_Date"] - h.groupby(dp.GROUP_COL)["Test_Date"].transform("min")).dt.days
fade = h.groupby(dp.GROUP_COL).apply(lambda s: pd.Series({
    "n_rpt": len(s), "span_days": s["days"].max(), "last_soh": s["SOH_Percent"].iloc[-1],
    "fade_pp_per_100_days": np.polyfit(s["days"], s["SOH_Percent"], 1)[0] * 100 if s["days"].nunique() > 1 else np.nan,
    "last_class": ev.REUSE_CLASS_NAMES[int(ev.reuse_class([s["SOH_Percent"].iloc[-1]])[0])],
}), include_groups=False).reindex(groups)
fade

## 5. Predictions from the E1 models and the baselines

Each model is fitted once on the pooled training periods (settings selected on validation), then predicts every group's test period. Baselines are per battery.

In [ ]:
features = dp.model_feature_columns(data)
predictions, fit_info = {}, {}
for model in CONFIG["models"]:
    preds, cfg = X.fit_predict(model, train, val, tests, features, CONFIG["seed"], CONFIG["neural_kwargs"], history=data)
    predictions[model] = np.concatenate([preds[g] for g in tests])
    fit_info[model] = {k: cfg.get(k) for k in ("alpha", "lr", "lr_at_grid_edge") if k in cfg}
for kind in ("persistence", "majority"):
    predictions[f"baseline_{kind}"] = np.concatenate([
        RU.baseline_predictions(P.make_task(data, g, CONFIG["train_frac"], CONFIG["val_frac"]).train, tests[g], kind)
        for g in tests])
print(fit_info)

## 6. Category agreement (primary cut-points)

In [ ]:
LABELS = {"ridge": "Ridge", "mlp": "MLP", "tft": "TFT", "baseline_persistence": "Persistence", "baseline_majority": "Majority"}
results = []
for name, pred in predictions.items():
    for unit in RU.UNITS:
        results.append(RU.evaluate(test_all, pred, unit).assign(model=name))
results = pd.concat(results, ignore_index=True)
cols = ["accuracy", "macro_f1", "n_classes_present", "kappa_linear", "off_by_more_than_one", "boundary_share", "soh_mae", "n"]
headline = results[(results[dp.GROUP_COL] == "ALL")].set_index(["unit", "model"])[cols]
print("Pooled over groups (headline unit: interval):")
display(headline.round(3))
per_group = results[(results["unit"] == "interval") & (results[dp.GROUP_COL] != "ALL")].pivot_table(
    index=dp.GROUP_COL, columns="model", values=["accuracy", "macro_f1"]).reindex(groups)
flags = results[(results["unit"] == "interval") & (results[dp.GROUP_COL] != "ALL") & (results["model"] == CONFIG["models"][0])].set_index(dp.GROUP_COL)["single_class"]
print("Per group (interval unit); single_class marks groups whose test period has one measured class:")
display(per_group.round(3).join(flags.rename(("", "single_class"))))

In [ ]:
shown = [*CONFIG["models"], "baseline_persistence"]
fig, axes = plt.subplots(1, len(shown), figsize=(3.6 * len(shown), 3.6), squeeze=False)
for k, (ax, name) in enumerate(zip(axes[0], shown)):
    conf = results[(results["model"] == name) & (results["unit"] == "interval") & (results[dp.GROUP_COL] == "ALL")]["confusion"].iloc[0]
    ax.imshow(conf, cmap="Blues", vmin=0, vmax=max(1, conf.max()))
    for (i, j), v in np.ndenumerate(conf):
        ax.text(j, i, int(v), ha="center", va="center", fontsize=9, color="white" if v > 0.6 * conf.max() else "#1a1a19")
    ax.set_xticks(range(4), CLASS_SHORT)
    ax.set_yticks(range(4), CLASS_SHORT if k == 0 else [])
    ax.set_xlabel("predicted class")
    if k == 0:
        ax.set_ylabel("measured class")
    ax.set_title(LABELS[name], loc="left")
    ax.grid(False)
fig.suptitle("Confusion matrices, pooled test-period RPT intervals")
plt.tight_layout()
plt.show()

## 7. Sensitivity to the cut-points (S-a1, S-a2)

Pooled interval-level agreement for every pre-registered cut-point variant. Large changes mean the category result depends on where the boundaries sit relative to the data, which must be discussed; the primary definition is not changed after seeing these numbers.

In [ ]:
sens = []
for variant, cuts in RU.CUTPOINT_VARIANTS.items():
    for name, pred in predictions.items():
        r = RU.evaluate(test_all, pred, "interval", cuts, by_group=False).iloc[0]
        sens.append({"variant": variant, "model": name, **{c: r[c] for c in ["accuracy", "macro_f1", "n_classes_present", "kappa_linear", "boundary_share"]}})
cutpoint_sensitivity = pd.DataFrame(sens).set_index(["variant", "model"])
cutpoint_sensitivity.round(3)

## 8. S-D3: end-of-day date convention

Labels are recomputed with the `"end"` convention (same-day cycles use the previous RPT), models are refitted on the relabelled data, and agreement is recomputed against the relabelled test periods.

In [ ]:
relabelled = P.relabel_date_only(data, health, "end")
r_train, r_val, r_tests = X.pooled_split(relabelled, groups, CONFIG["train_frac"], CONFIG["val_frac"])
r_test_all = pd.concat(r_tests.values(), ignore_index=True)
d3 = []
for model in CONFIG["models"]:
    preds, _ = X.fit_predict(model, r_train, r_val, r_tests, features, CONFIG["seed"], CONFIG["neural_kwargs"], history=relabelled)
    r = RU.evaluate(r_test_all, np.concatenate([preds[g] for g in r_tests]), "interval", by_group=False).iloc[0]
    primary = headline.loc[("interval", model)]
    d3.append({"model": model, "accuracy_start": primary["accuracy"], "accuracy_end": r["accuracy"],
               "macro_f1_start": primary["macro_f1"], "macro_f1_end": r["macro_f1"]})
date_convention_sensitivity = pd.DataFrame(d3).set_index("model")
print(f"relabelled: {int(relabelled['label_changed'].sum())} labels changed, {len(data) - len(relabelled)} rows dropped")
date_convention_sensitivity.round(3)

## 9. Save results

In [ ]:
to_save = results.drop(columns=["confusion"]).assign(confusion=[c.tolist() for c in results["confusion"]])
out = X.write_results(RESULTS_DIR / "reuse_assessment.json", {
    "profile": PROFILE, "config": CONFIG, "fit_info": fit_info,
    "class_distribution": {"all": dist_all.reset_index(), "test": dist_test.reset_index()},
    "fade_rate": fade.reset_index(), "results": to_save,
    "cutpoint_sensitivity": cutpoint_sensitivity.reset_index(),
    "date_convention_sensitivity": date_convention_sensitivity.reset_index(),
}, REPO_ROOT)
print("wrote", out.relative_to(REPO_ROOT))
if PROFILE != "paper":
    print("NOTE: smoke profile — these numbers are for code checks only.")

## 10. Limitations to state with these results

* **Classes describe change, not suitability.** They measure capacity fade relative to the first observed RPT, not absolute health or fitness for an application. Application labels appear only in S-b, and only if the data provenance is confirmed.
* **Cut-points are conventions**, fixed before evaluation (D10); no verified standard gives numerical second-life class boundaries.
* **Capacity is the only criterion.** Resistance, power fade and safety screening are not assessed.
* **Small sample.** Six cells (if the provenance is confirmed, one cell model), so results do not establish generalisation across chemistries or manufacturers.
* **Label resolution.** Labels change only at RPTs; boundary share shows how many samples sit closer to a cut-point than the model's error.